# Deep Burst SR Colab Starter

This notebook mounts Google Drive, prepares a synthetic RGB dataset from DIV2K, trains the model, and writes checkpoints plus sample outputs to Drive.

In [ ]:
import os
IN_COLAB = 'COLAB_GPU' in os.environ
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Not running in Colab: skipping Google Drive mount.')


In [ ]:
from pathlib import Path

if IN_COLAB:
    %cd /content
    REPO_ROOT = Path('/content/deep_burst_sr_reimplementation')
    if not REPO_ROOT.exists():
        !git clone https://github.com/Karan2005-sys/deep_burst_sr_reimplementation.git {REPO_ROOT}
else:
    REPO_ROOT = Path.cwd()

%cd {REPO_ROOT}
!pip install -r {REPO_ROOT / 'requirements.txt'}
print('Using repo:', REPO_ROOT)


In [ ]:
if IN_COLAB:
    SYNTH_ROOT = '/content/drive/MyDrive/SyntheticRGB'
    REAL_ROOT = '/content/drive/MyDrive/BurstSR'
    OUTPUT_ROOT = '/content/drive/MyDrive/dbsr_outputs'
else:
    SYNTH_ROOT = '/tmp/SyntheticRGB'
    REAL_ROOT = '/tmp/BurstSR'
    OUTPUT_ROOT = '/tmp/dbsr_outputs'

SYNTH_CONFIG = 'configs/synthetic_rgb.yaml'
REAL_CONFIG = 'configs/real_burstsr.yaml'

print('SYNTH_ROOT:', SYNTH_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)


In [ ]:
# Verify datasets package version without shell heredocs
from pathlib import Path
import subprocess
import sys

req_text = Path('requirements.txt').read_text(encoding='utf-8-sig').splitlines()
datasets_req = next((line.strip() for line in req_text if line.strip().startswith('datasets')), 'datasets (not pinned)')
print('requirements.txt entry:', datasets_req)

show = subprocess.run([sys.executable, '-m', 'pip', 'show', 'datasets'], capture_output=True, text=True)
if show.returncode != 0:
    print('datasets is not installed yet.')
else:
    for line in show.stdout.splitlines():
        if line.startswith(('Name:', 'Version:', 'Location:')):
            print(line)


## Prepare synthetic RGB data from DIV2K
This creates `train/` and `val/` under `SYNTH_ROOT` automatically using DIV2K HR images.

In [ ]:
!mkdir -p {OUTPUT_ROOT}
!python {REPO_ROOT / 'scripts' / 'prepare_div2k_synthetic.py'} --output-root {SYNTH_ROOT}


In [ ]:
!python {REPO_ROOT / 'scripts' / 'train.py'} --config {REPO_ROOT / SYNTH_CONFIG} --data-root {SYNTH_ROOT} --output-dir {OUTPUT_ROOT}/synthetic_run


In [ ]:
!find {OUTPUT_ROOT}/synthetic_run -maxdepth 3 -type f | sort | tail -n 20

In [ ]:
from IPython.display import display
from PIL import Image
from pathlib import Path

visual_root = Path(OUTPUT_ROOT) / 'synthetic_run' / 'visuals'
epoch_dirs = sorted([p for p in visual_root.glob('epoch_*') if p.is_dir()])
latest = epoch_dirs[-1] if epoch_dirs else None
print('Latest preview folder:', latest)
if latest is not None:
    display(Image.open(latest / 'prediction.png'))
    display(Image.open(latest / 'target.png'))

## Later: switch to real BurstSR
When the real dataset is available, point `REAL_ROOT` to it and run the command below.

In [ ]:
# !python {REPO_ROOT / 'scripts' / 'train.py'} --config {REPO_ROOT / REAL_CONFIG} --data-root {REAL_ROOT} --output-dir {OUTPUT_ROOT}/real_run
